# 第5课 · 给每个频率发一张身份证——复数（complex number）的模与相位（phase），FFT 的母语

**学习目标**
1. 理解复数 `a+bj` 的直角坐标（rectangular coordinates / Cartesian coordinates）与极坐标（polar coordinates）表示
2. 实现 `magnitude_phase(z)` 返回 `(模（magnitude，r）, 相位（phase，φ）)` 元组
3. 理解 FFT 输出为什么是复数：模 = 该频率（frequency）有多响，相位 = 时间对齐
4. 为 L06 欧拉公式（Euler's formula） `e^{jθ} = cos(θ) + j·sin(θ)` 打好基础

**来自 L04 的记忆**：`sinusoid` 的振幅（amplitude） A 和相位 φ，
在 FFT 输出里分别对应一个复数的**模**和**相位**。
本课让你亲手从复数中读出这两个量。

**Aurora 连接**：`magnitude_spectrogram` = 对 FFT 每个频点（frequency bin）的复数取模；
相位在声码器（vocoder）和波形重建（L45）里至关重要。

← **上一课**　[L04 · 正弦波三要素](L04_trig.ipynb)

> 上节课学习了 **正弦波三要素**：频率决定音高、振幅决定响度、相位决定起点，亲手实现。  
> 本课将探讨 **复数的模与相位**：给每个频率发身份证，FFT 的母语。

## 本课剧情：给每个频率分量办一张"身份证"

FFT 分析声音时，每一个频率分量都会得到一个复数 `X[k] = a + bj`。

但"复数"到底是什么？它不是虚构的——它是一张**二维身份证**，同时记录了两件事：
- **这个频率有多响**（振幅 / 模）
- **这个频率从哪里起步**（相位）

一张身份证，两个信息，一次打包。

想象你在黑板上画一个箭头。箭头的**长度**就是振幅，箭头的**方向**就是相位。这个箭头就是复数，用极坐标写是 `r·e^{iθ}`，用直角坐标写是 `a + bj`。

| 描述方式 | 记录什么 | 公式 |
|---|---|---|
| 直角坐标（a+bj）| 实部 a + 虚部 b | a = r·cos(θ)，b = r·sin(θ) |
| 极坐标（r, θ）| 模 r + 相位 θ | r = √(a²+b²)，θ = arctan2(b, a) |

本课你将实现 `magnitude_phase(z)`，从复数直接读出这张"身份证"的两面。

## 🤔 先想一想：不用复数，描述一个频率要几个数字？

**先别看公式，想 30 秒。** 假设你只能用实数，要完整交代"某个频率分量"，你至少得写下几个数？

- 写一个**振幅**？——不够。两个一样响的音，可能"起步时刻"不一样。
- 那再补一个**相位**。于是最少要 **两个数**：一个管多响，一个管从哪起步。
- 换个等价写法：`cos` 前面一个系数 + `sin` 前面一个系数，还是 **两个自由度**。

一个频率天生带着**两个自由度**。而一个复数 `a + bj` 恰好装两个数——一半管模（振幅）、一半管相位，严丝合缝。

> 所以复数不是数学炫技，而是一门**工程语言**：它把"描述一个频率所需的两个数"打包成一个能直接做乘法、做旋转的对象。这正是为什么 FFT 把复数当母语。

## ⚠️ 常见误解 (Common Pitfall)

> 不要把复数理解成"虚构 / 不存在的数字"。它其实是**"模 + 相位"两个真实自由度的打包**——一个箭头的长度和方向，两样都看得见、量得出。"虚"字只是历史遗留的坏名字。

## 1. 复数的基本操作

有没有想过：为什么数学家要发明一个"虚数单位" i（Python 用 j）？

原来，实数轴只有一个维度。乘以 -1 可以"翻转方向"，但翻转不够用——有些旋转需要**转 90°**而不是 180°。i 的定义就是"乘以 i = 逆时针旋转 90°"：

```
i² = -1（转了两次 90°= 转了 180°= 乘以 -1）
```

所以复数 `a + bj` 不是两个无关的数，而是一个**二维平面上的点**：
- 实部 a → 水平方向
- 虚部 b → 垂直方向

Python 原生支持复数；NumPy 的 `np.exp(1j * theta)` 直接给出单位圆上角度 θ 对应的点。

## 实验入口：角度、坐标和旋转

角度 θ 对应的单位复数是 `cos(θ) + sin(θ)j`：实部是水平分量，虚部是垂直分量，模恒为 1。下面遍历一圈，验证极坐标与直角坐标的互换关系。

In [ ]:
import numpy as np
z = 3 + 4j
print('实部:', z.real, ' 虚部:', z.imag)
print('模 |z| =', abs(z), ' (=√(3²+4²)=5)')
print('相位(弧度) =', np.angle(z))

## 动手观察：复数就是"旋转的位置"

观察下表的 `real` 和 `imag` 列：角度每增加 π/2，坐标在单位圆（unit circle）上逆时针移一步。`magnitude` 列始终为 1，改变的只有 `phase`。

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('角度 =', np.round(angles, 3))
print('实部 cos =', np.round(z.real, 3))
print('虚部 sin =', np.round(z.imag, 3))
print('复数 z =', np.round(z, 3))


## 代码实验：旋转一整圈

从 0 到 2π 均匀取 8 个角度，打印每个点的实部、虚部、模和相位，验证单位圆上模始终为 1、相位就是角度本身。

In [ ]:
import numpy as np

for k in range(9):
    theta = 2 * np.pi * k / 8
    z = np.exp(1j * theta)
    print(f'k={k} theta={theta:.2f} -> ({z.real:+.3f}, {z.imag:+.3f})')


## 2. ✏️ 实现 `magnitude_phase(z)` 返回 `(模, 相位)`

**推理路线**：
1. 模是复数到原点的距离：`r = √(real² + imag²)`，Python 内置 `abs(z)` 直接返回此值，对标量和 numpy 数组均适用。
2. 相位是与 x 轴正半轴的夹角：`θ = arctan2(imag, real)`，`np.angle(z)` 封装了此计算，输出范围 `(-π, π]`，单位弧度（radian）。
3. 返回元组 `(magnitude, phase)` 是因为 FFT 后处理通常同时需要两者，比分开调用两次更高效。

**参考输入输出**：`z = 3 + 4j` → 模 = 5.0，相位 ≈ 0.9273 弧度（约 53°，即 arctan(4/3)）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `magnitude_phase` 前明确三件事：
- 输入：复数 `z`（Python 或 numpy 复数均可）
- 关键步骤：`abs(z)` 求模；`np.angle(z)` 求相位，范围 (−π, π]，单位弧度
- 返回：元组 `(magnitude, phase)`，两者均为浮点数

### 📐 手动实现规则（先自己做，再看提示）

**第一步（必做）**：仅用下列工具填写 stub：
- `z.real`，`z.imag` — 读取实部、虚部
- `np.sqrt()` — 开根号
- `np.arctan2(b, a)` — 反正切，返回 (-π, π]

**不允许** 在 stub 里直接调用 `abs(z)` 或 `np.angle(z)`（那等于零实现）。

通过断言后，在函数末尾加注释：`# 等价于: return abs(z), np.angle(z)` — 对比两种写法结果是否完全一致。


In [ ]:
def magnitude_phase(z):
    # ✏️ TODO: 仅用 z.real, z.imag, np.sqrt, np.arctan2 实现（不允许直接调 abs/np.angle）
    raise NotImplementedError("请实现 magnitude_phase(z)：用 z.real, z.imag, np.sqrt, np.arctan2")


In [ ]:
import numpy as np
try:
    mag, ph = magnitude_phase(3 + 4j)
    print('模 =', mag, ' 相位 =', round(ph, 4))
    assert abs(mag - 5.0) < 1e-9, f"模应为 5.0，实际得到 {mag}"
    assert abs(ph - np.arctan2(4, 3)) < 1e-9, f"相位应为 {np.arctan2(4,3):.4f}，实际得到 {ph}"
    print('✅ 通过：你能从复数读出模和相位了。')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️ 还未实现：{e}')
    print('  请回到上方 magnitude_phase 函数，用 z.real/z.imag/np.sqrt/np.arctan2 填写实现。')


**🔗 Aurora 连接**：`magnitude_spectrogram` 取的就是 FFT 复数结果的模；相位在声码器/重建里很关键。下一课：把复数和三角连起来——欧拉公式。

In [ ]:
for k in range(8):
    theta = 2*np.pi*k/8
    z = np.exp(1j*theta)
    radius = abs(z)
    print(f'k={k} | z=({z.real:+.2f}, {z.imag:+.2f}) | 半径={radius:.2f}')
print('所有点半径都接近 1，说明它们都在单位圆上。')


## 参数实验：单位圆上 8 点的模与相位

构造 `z = np.exp(1j * np.linspace(0, 2*np.pi, 8))`，打印每个点的模和相位，确认模恒为 1.0，相位以 π/4 等步长均匀分布；注意 `np.angle` 的输出范围是 (−π, π]，超过 π 的角度（5π/4、3π/2、7π/4）会折返显示为负值（−3π/4、−π/2、−π/4）。

再把 `3 + 4j` 换成 `3 - 4j`，观察相位从 +0.9273 变为 -0.9273：虚部取反即相位取反，模不变——验证模与相位相互独立。

In [ ]:
import numpy as np

# 实验 A：单位圆上 8 点的模与相位
z8 = np.exp(1j * np.linspace(0, 2 * np.pi, 8, endpoint=False))
print('单位圆 8 点（endpoint=False，避免重复 0/2π）')
print(f'  模：{np.abs(z8).round(4)}')  # 全部应为 1.0
print(f'  相位（弧度）：{np.angle(z8).round(4)}')  # 0, π/4, ..., π，随后折返为 -3π/4, -π/2, -π/4（np.angle 范围 (-π, π]）
assert np.allclose(np.abs(z8), 1.0), '模应全部为 1.0'
print('✅ 模恒为 1，相位均匀分布')

# 实验 B：虚部取反 → 相位取反，模不变
z_pos = 3 + 4j
z_neg = 3 - 4j
print(f'\n3+4j：模={abs(z_pos):.4f}，相位={np.angle(z_pos):.4f}')
print(f'3-4j：模={abs(z_neg):.4f}，相位={np.angle(z_neg):.4f}')
assert abs(abs(z_pos) - abs(z_neg)) < 1e-9, '模应相等'
assert abs(np.angle(z_pos) + np.angle(z_neg)) < 1e-9, '相位应互为相反数'
print('✅ 虚部取反 = 相位取反，模不变')


## FFT 输出预览：每个频点都是一个复数

不用理解推导——只看输出的数据类型和值，建立「FFT 输出 = 复数数组」的直觉。

In [ ]:
import numpy as np

# 把 L04 的和弦信号做 FFT（Module 5 会从零实现；这里只看输出形状）
sr, dur = 200, 1.0
t = np.arange(round(dur * sr)) / sr
chord = (np.sin(2*np.pi*20*t) +
         np.sin(2*np.pi*32*t) +
         np.sin(2*np.pi*40*t)) / 3  # 三个频率

X = np.fft.rfft(chord)  # FFT 输出：复数数组
freqs = np.fft.rfftfreq(len(chord), 1/sr)

print(f'输入：{chord.shape}  float64')
print(f'输出：{X.shape}  {X.dtype}  ← 复数！')
print()

# 找三个峰值频率，打印对应复数
magnitudes = np.abs(X)
top3 = np.argsort(magnitudes)[-3:][::-1]
print(f'  频率 (Hz)  |  复数 X[k]              |  模（响度）|  相位（弧度）')
print('-' * 70)
for idx in sorted(top3):
    z = X[idx]
    mag, phase = abs(z), np.angle(z)
    print(f'  {freqs[idx]:>8.1f}  |  {z.real:+.2f} + {z.imag:+.2f}j  |  {mag:.2f}        |  {phase:.3f}')

print()
print('模 ≈ 33（振幅 1/3 × 采样点数/2 = 1/3 × 100 ≈ 33）；归一化原因（Parseval 定理）将在 L40 推导，现在接受这个数字即可。')
print('相位 ≈ -π/2，因为 sin 波在 t=0 时值为 0，cos 相位领先 π/2。')

## 🎯 未来的回报 (Future Payoff)

今天你亲手从复数里读出的**模与相位**，会在 **L06 欧拉公式 / L35、L37–L44 的 FFT 与 STFT / L70 Whisper** 一再出现——那时每个频点都是一个复数，模就是"这个频率多响"，相位就是"它怎么对齐"。今天搞懂它，后面就不必再从头猜。

## 本课收束

现在你能用 `magnitude_phase(z)` 从任意复数读出模（`abs(z)`）
和相位（`np.angle(z)`）。

**关键连接**：
- L04 的 `sinusoid(t, A=r, f=f0, phi=θ)` ↔ FFT 输出复数 `X[k]` 的 `(模=r, 相位=θ)`
- `magnitude_spectrogram = |FFT 输出|`（逐元素取模，L40 实现）
- 相位谱（phase spectrum）在波形重建（L45 STFT 逆变换）中不可或缺

**下一课 L06**：欧拉公式 `e^{jθ} = cos(θ) + j·sin(θ)`——
把今天的极坐标和上节课的正弦波直接连起来，是理解 DFT 矩阵（L37）的最短路径。

## ✏️ 白板挑战：复数手算（目标 8 分钟）

盖上屏幕，纸上作答：

**复数**：z = 3 + 4j

**问 1**：z 的模（magnitude）是多少？公式是 |z| = √(a² + b²)。

**问 2**：z 的相位（phase）是多少弧度？公式 θ = arctan2(b, a)（先手算，再运行对答案格）。

**问 3**：把 z 改写成极坐标形式 r·e^{iθ}，写出 r 和 θ 的数值。

**问 4**：`np.exp(1j * np.pi)` 等于多少？（Euler 公式：e^{iπ} = cos(π) + i·sin(π) = ?）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

z = 3 + 4j

# 问1：模
mag_theory = 5.0  # √(9+16) = √25 = 5
mag_code = abs(z)
assert np.isclose(mag_code, mag_theory, atol=1e-10)
print(f"Q1 ✅  |3+4j| = √(9+16) = 5  代码值 {mag_code:.4f}")

# 问2：相位
ph_theory = np.arctan2(4, 3)  # ≈ 0.9273 rad
ph_code = np.angle(z)
assert np.isclose(ph_code, ph_theory, atol=1e-10)
print(f"Q2 ✅  ∠(3+4j) = arctan2(4,3) ≈ {ph_theory:.4f} rad ({np.degrees(ph_theory):.2f}°)")

# 问3：极坐标
r, theta = mag_code, ph_code
z_polar = r * np.exp(1j * theta)
assert np.isclose(z_polar.real, z.real, atol=1e-10)
assert np.isclose(z_polar.imag, z.imag, atol=1e-10)
print(f"Q3 ✅  z = {r:.1f}·e^{{i·{theta:.4f}}} = {z_polar.real:.1f} + {z_polar.imag:.1f}j (验证重合)")

# 问4：Euler 公式
z_euler = np.exp(1j * np.pi)
assert np.isclose(z_euler.real, -1.0, atol=1e-10)
assert np.isclose(z_euler.imag,  0.0, atol=1e-10)
print(f"Q4 ✅  e^{{iπ}} = cos(π) + i·sin(π) = -1 + 0j  代码值 {z_euler:.4f}")
print("\n🎉 复数白板挑战通过！模和相位就是这张频率'身份证'的两面。")

In [ ]:
# ✏️ 本课自评
l05_review = {
    "magnitude_phase_implemented": None,  # magnitude_phase 实现并通过断言？True/False
    "rectangular_polar_understood": None, # 能互相转换直角坐标↔极坐标？True/False
    "euler_formula_intuition":      None, # 理解 e^{iθ} 就是单位圆上旋转θ的点？True/False
    "whiteboard_passed":            None, # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l05_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l05_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L05 全部通关！进入 L06：欧拉公式 e^{iθ}=cosθ+isinθ')

---

→ **下一课**　[L06 · 欧拉公式 e^{iθ}=cosθ+isinθ](L06_euler.ipynb)

> 下节课将学习 **欧拉公式 e^{iθ}=cosθ+isinθ**：旋转因子是 FFT 的命根子。